In [8]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'    # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [5]:
reviews = pd.read_csv('../../../data/processed/steam_indie_reviews.csv')
games = pd.read_csv('../../../data/processed/stratified_games_early_reviews.csv')
histogram = pd.read_csv('../../../data/processed/steam_review_histogram.csv')

print(f"games: {games.shape}")
print(f"reviews: {reviews.shape}")
print(f"histogram: {histogram.shape}")

games: (74, 5)
reviews: (13106, 21)
histogram: (6003, 9)


# 1. 오래 플레이한 유저일수록 긍정 리뷰를 남길까?

In [9]:
playtime_hour = reviews['author_playtime_at_review'] / 60
p99 = playtime_hour.quantile(0.99)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    '플레이타임 분포 (전체)',
    f'플레이타임 분포 (상위 1% 제외, ≤ {p99:.0f}h)'
))

fig.add_trace(go.Histogram(x=playtime_hour, nbinsx=100, name='전체'), row=1, col=1)
fig.add_trace(go.Histogram(x=playtime_hour[playtime_hour <= p99], nbinsx=100, name='1% 제외'), row=1, col=2)

fig.update_xaxes(title_text='플레이타임 (시간)')
fig.update_yaxes(title_text='리뷰 수')
fig.update_layout(height=450, showlegend=False)
fig.show()

print(playtime_hour.describe().apply(lambda x: f"{x:.1f}h"))

count    13106.0h
mean        23.4h
std        123.9h
min          0.1h
25%          2.1h
50%          5.4h
75%         14.9h
max       4398.8h
Name: author_playtime_at_review, dtype: str


In [10]:
# 출시 3개월 이내 리뷰 작성 유저의 작성 시점 플레이타임 추출
# (playtime 정보는 histogram이 아닌 reviews에 있음)

release_dates = histogram[['appid', 'release_date']].drop_duplicates()
release_dates['release_date'] = pd.to_datetime(release_dates['release_date'])
release_dates['cutoff_date'] = release_dates['release_date'] + pd.DateOffset(months=3)

reviews['review_date'] = pd.to_datetime(reviews['timestamp_created'], unit='s')

reviews_with_cutoff = reviews.merge(release_dates[['appid', 'cutoff_date']], how='inner', on='appid')
reviews_3m = reviews_with_cutoff[reviews_with_cutoff['review_date'] <= reviews_with_cutoff['cutoff_date']]

# appid별 작성 시점 플레이타임 통계를 파생 컬럼으로 생성
playtime_stats = reviews_3m.groupby('appid')['author_playtime_at_review'].agg(
    playtime_mean='mean',
    playtime_median='median',
).reset_index()
playtime_stats['playtime_mean_hour']   = (playtime_stats['playtime_mean']   / 60).round(1)
playtime_stats['playtime_median_hour'] = (playtime_stats['playtime_median'] / 60).round(1)

games = games.merge(playtime_stats[['appid', 'playtime_mean_hour', 'playtime_median_hour']], how='left', on='appid')

print(games.shape)
games.head()

(74, 7)


,appid,name_store,owners_lower,early_review_count,positive_rate_3m,playtime_mean_hour,playtime_median_hour
0,1432860,Sun Haven,500000,8174,72.0,37.0,27.0
1,1473350,(the) Gnorp Apologue,200000,3999,92.0,32.9,13.0
2,1993150,轮回修仙路,200000,1994,69.0,39.3,24.8
3,2527500,MiSide,1000000,91831,100.0,14.7,6.1
4,1169040,Necesse,1000000,23053,79.0,62.9,42.7


In [11]:
# 게임별 플레이타임 중앙값 분포 확인
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=games['playtime_median_hour'],
    nbinsx=30,
    hovertemplate='%{x:.1f}h: %{y}개 게임<extra></extra>'
))

median_val = games['playtime_median_hour'].median()
fig.add_vline(x=median_val, line_dash='dash', line_color='red',
              annotation_text=f'중앙값 {median_val:.1f}h', annotation_position='top right')

fig.update_xaxes(title_text='플레이타임 중앙값 (시간/게임)')
fig.update_yaxes(title_text='게임 수')
fig.update_layout(title='게임별 초기 3개월 리뷰 작성 시점 플레이타임 중앙값 분포', height=430)
fig.show()

print(games['playtime_median_hour'].describe().apply(lambda x: f"{x:.1f}h"))

count    73.0h
mean      9.7h
std      13.0h
min       0.9h
25%       3.2h
50%       6.2h
75%      11.7h
max      95.6h
Name: playtime_median_hour, dtype: str
